# Lesson 20: Feature Engineering, Pipelines & Final Project

*ML Academy — companion notebook*

Clean and encode data, build leak-proof pipelines, and complete an end-to-end project.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-20.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 9. Final project: predicting customer churn

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 2000
df = pd.DataFrame({
    "tenure_months": rng.integers(1, 73, n),
    "monthly_charge": rng.normal(65, 20, n).clip(15, 130).round(2),
    "support_calls": rng.poisson(1.5, n),
    "contract": rng.choice(["month-to-month", "one-year", "two-year"], n, p=[0.55, 0.25, 0.20]),
    "payment": rng.choice(["card", "bank", "check"], n, p=[0.4, 0.35, 0.25]),
    "city": rng.choice(["Chennai", "Mumbai", "Delhi", "Pune"], n),
})
df["total_spent"] = (df["tenure_months"] * df["monthly_charge"] * rng.normal(1, 0.05, n)).round(2)
score = (-2.0 + 1.6 * (df["contract"] == "month-to-month") - 0.035 * df["tenure_months"]
         + 0.018 * df["monthly_charge"] + 0.45 * df["support_calls"] + 0.5 * (df["payment"] == "check"))
df["churned"] = (rng.random(n) < 1 / (1 + np.exp(-score))).astype(int)
for col, frac in [("monthly_charge", 0.05), ("payment", 0.04)]:     # real data has gaps
    df.loc[rng.random(n) < frac, col] = np.nan

print(df.shape)
print(df.head(3).to_string())
print("churn rate:", round(df["churned"].mean(), 3))
print(df.isna().sum()[df.isna().sum() > 0].to_dict())

(2000, 8)
   tenure_months  monthly_charge  support_calls        contract payment     city  total_spent  churned
0              7           50.16              2        one-year    bank     Pune       316.79        0
1             56           83.49              0  month-to-month     NaN    Delhi      4578.37        1
2             48           65.69              2  month-to-month    card  Chennai      3111.67        1
churn rate: 0.403
{'monthly_charge': 105, 'payment': 86}


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="churned")
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

def add_features(X):
    X = X.copy()
    X["charge_per_call"] = X["monthly_charge"] / (X["support_calls"] + 1)   # a ratio feature
    X["log_total_spent"] = np.log1p(X["total_spent"])                       # tame the skew
    return X

numeric = ["tenure_months", "monthly_charge", "support_calls", "total_spent",
           "charge_per_call", "log_total_spent"]
categorical = ["contract", "payment", "city"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

candidates = {
    "baseline (majority)": DummyClassifier(strategy="most_frequent"),
    "logistic regression": LogisticRegression(max_iter=1000),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=0),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
for name, clf in candidates.items():
    pipe = Pipeline([("features", FunctionTransformer(add_features)),
                     ("prep", preprocess), ("model", clf)])
    s = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name:22s} ROC-AUC {s.mean():.3f} ± {s.std():.3f}")

baseline (majority)    ROC-AUC 0.500 ± 0.000


logistic regression    ROC-AUC 0.771 ± 0.008


random forest          ROC-AUC 0.756 ± 0.010


gradient boosting      ROC-AUC 0.739 ± 0.008


In [5]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([("features", FunctionTransformer(add_features)), ("prep", preprocess),
                 ("model", LogisticRegression(max_iter=1000))])
search = GridSearchCV(pipe, {"model__C": [0.01, 0.1, 1, 10]}, cv=5, scoring="roc_auc")
search.fit(X_train, y_train)
print("best C:", search.best_params_["model__C"], " CV ROC-AUC:", round(search.best_score_, 3))

best C: 0.1  CV ROC-AUC: 0.772


In [6]:
from sklearn.metrics import roc_auc_score, classification_report

best = search.best_estimator_
proba = best.predict_proba(X_test)[:, 1]
print("test ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, (proba >= 0.5).astype(int), target_names=["stayed", "churned"]))

test ROC-AUC: 0.765
              precision    recall  f1-score   support

      stayed       0.73      0.82      0.77       239
     churned       0.67      0.55      0.61       161

    accuracy                           0.71       400
   macro avg       0.70      0.68      0.69       400
weighted avg       0.71      0.71      0.70       400



In [7]:
names = best.named_steps["prep"].get_feature_names_out()
coefs = best.named_steps["model"].coef_[0]
for name, c in sorted(zip(names, coefs), key=lambda t: -abs(t[1]))[:5]:
    print(f"{name:32s} {c:+.2f}")

cat__contract_month-to-month     +0.85
num__tenure_months               -0.59
num__support_calls               +0.45
cat__contract_one-year           -0.43
cat__contract_two-year           -0.42


In [8]:
import joblib

joblib.dump(best, "churn_model.joblib")
reloaded = joblib.load("churn_model.joblib")       # only ever load files you trust!

new_customer = pd.DataFrame([{"tenure_months": 3, "monthly_charge": 95.0, "support_calls": 4,
                              "contract": "month-to-month", "payment": "check", "city": "Pune",
                              "total_spent": 285.0}])
print("churn probability:", round(reloaded.predict_proba(new_customer)[0, 1], 3))

churn probability: 0.958


## 15. Going deeper: leak-proof pipelines and efficient tuning

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import TargetEncoder

# A category with 2,000 levels (like a postcode) that has NO real link to the target
rng = np.random.default_rng(0)
n = 5000
code = rng.integers(0, 2000, n)
y = rng.integers(0, 2, n)
cv = KFold(5, shuffle=True, random_state=0)

# Wrong: encode with each level's average target computed on ALL the data, then cross-validate
means = pd.Series(y).groupby(code).mean()
leaky = means[code].to_numpy()[:, None]
print("naive target encoding, CV accuracy:", round(cross_val_score(LogisticRegression(), leaky, y, cv=cv).mean(), 3))

# Right: TargetEncoder inside the pipeline (it also uses internal cross-fitting when training)
pipe = make_pipeline(TargetEncoder(cv=KFold(5, shuffle=True, random_state=0)), LogisticRegression())
print("target encoding inside the pipeline, CV accuracy:", round(cross_val_score(pipe, code[:, None], y, cv=cv).mean(), 3))

naive target encoding, CV accuracy: 0.731


target encoding inside the pipeline, CV accuracy: 0.497


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

# One pipeline for mixed data: numbers, a category and free text (Chennai Homes)
df = pd.read_csv("../data/houses.csv")
df.loc[2, "age_yrs"] = np.nan                              # pretend one value is missing
prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["size_m2", "bedrooms", "dist_km", "age_yrs"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["area"]),
    ("txt", TfidfVectorizer(), "description"),
])
model = make_pipeline(prep, Ridge(alpha=1.0))
scores = cross_val_score(model, df, df["price_lakh"], cv=KFold(4, shuffle=True, random_state=0), scoring="neg_mean_absolute_error")
print("columns after preprocessing:", prep.fit_transform(df).shape[1])
print("4-fold mean absolute error:", round(-scores.mean(), 1), "lakh")

columns after preprocessing: 30
4-fold mean absolute error: 7.0 lakh


In [11]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from scipy.stats import loguniform, randint

Xb, yb = load_breast_cancer(return_X_y=True)
gb = HistGradientBoostingClassifier(random_state=0)
grid = GridSearchCV(gb, {"learning_rate": [0.03, 0.1, 0.3], "max_leaf_nodes": [7, 15, 31],
                         "l2_regularization": [0.0, 1.0]}, cv=5).fit(Xb, yb)
rand = RandomizedSearchCV(gb, {"learning_rate": loguniform(0.01, 0.5), "max_leaf_nodes": randint(4, 64),
                               "l2_regularization": loguniform(1e-3, 10)}, n_iter=18, cv=5, random_state=0).fit(Xb, yb)
print(f"grid search:   {len(grid.cv_results_['params'])} settings, best CV accuracy {grid.best_score_:.3f}")
print(f"random search: {len(rand.cv_results_['params'])} settings, best CV accuracy {rand.best_score_:.3f}")

grid search:   18 settings, best CV accuracy 0.975
random search: 18 settings, best CV accuracy 0.972


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [12]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
df.loc[3, "age_yrs"] = np.nan                     # pretend one value is missing
num = ["size_m2", "bedrooms", "dist_km", "age_yrs"]
numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
with_area = ColumnTransformer([("num", numeric, num), ("cat", OneHotEncoder(handle_unknown="ignore"), ["area"])])
numbers_only = ColumnTransformer([("num", numeric, num)])
for name, prep in [("numbers only", numbers_only), ("numbers + area", with_area)]:
    model = Pipeline([("prep", prep), ("ridge", Ridge(alpha=1.0))])
    mae = -cross_val_score(model, df, df["price_lakh"], cv=4, scoring="neg_mean_absolute_error").mean()
    print(f"{name:15s} cross-validated MAE: {mae:.1f} lakh")

numbers only    cross-validated MAE: 6.9 lakh
numbers + area  cross-validated MAE: 7.2 lakh


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [13]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Standardise with training statistics

Standardise the new value using the training mean and standard deviation.

In [14]:
import numpy as np
train = np.array([10.0, 20.0, 30.0])
new = 40.0
z = None  # ← your code here
check(z, 2.449489742783178, hint='(new − training mean) / training standard deviation.')

❌ Not yet. Hint: (new − training mean) / training standard deviation.


### Challenge 2: One-hot encoding

One-hot encode the colour column with pandas and count the columns produced.

In [15]:
import pandas as pd
df = pd.DataFrame({"colour": ["red", "green", "blue", "green"]})
onehot = df[["colour"]]  # ← replace with one-hot encoding
n_cols = onehot.shape[1]
check(n_cols, 3, hint="pd.get_dummies(df['colour']) makes one column per category.")

❌ Not yet. Hint: pd.get_dummies(df['colour']) makes one column per category.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Standardise with training statistics** — the missing line:

```python
z = (new - train.mean()) / train.std()
```

**2. One-hot encoding** — the missing line:

```python
onehot = pd.get_dummies(df["colour"])
```